In [ ]:
"""
Standalone example demonstrating OData query validator usage.

This script can be run independently to test the validator without
any LangGraph or agent infrastructure.

Usage:
    python standalone_example.py
"""

from odata_query_validator import ODataQueryValidator
from validator_config import ValidatorConfig
from validation_result import ValidationSeverity


def print_result(query: str, result):
    """Pretty print validation results."""
    print("\n" + "="*80)
    print(f"Query: {query}")
    print("="*80)
    print(f"Status: {'✓ VALID' if result.is_valid else '✗ INVALID'}")
    
    if result.estimated_cost_score is not None:
        print(f"Cost Score: {result.estimated_cost_score:.1f}/100")
    
    if result.estimated_row_count is not None:
        print(f"Estimated Rows: {result.estimated_row_count}")
    
    if result.issues:
        print(f"\nIssues Found: {len(result.issues)}")
        print("-"*80)
        
        for i, issue in enumerate(result.issues, 1):
            severity_icon = {
                ValidationSeverity.CRITICAL: "🔴",
                ValidationSeverity.WARNING: "🟡",
                ValidationSeverity.INFO: "🔵"
            }.get(issue.severity, "⚪")
            
            print(f"\n{i}. {severity_icon} [{issue.severity.upper()}] {issue.category}")
            print(f"   {issue.message}")
            if issue.query_segment:
                print(f"   Segment: {issue.query_segment}")
            if issue.suggestion:
                print(f"   💡 Suggestion: {issue.suggestion}")
    
    if not result.is_valid:
        print("\n" + "-"*80)
        print("Error Message:")
        print(result.get_error_message())


def example_1_valid_queries():
    """Example 1: Valid queries that pass all checks."""
    print("\n" + "🟢 "*40)
    print("EXAMPLE 1: Valid Queries")
    print("🟢 "*40)
    
    validator = ODataQueryValidator(config=ValidatorConfig.for_production())
    
    valid_queries = [
        "Users?$top=10&$select=Name,Email",
        "Orders?$top=50&$filter=TotalAmount gt 100&$orderby=OrderDate desc",
        "Products?$top=20&$select=Name,Price&$filter=Category eq 'Electronics'",
        "Customers?$top=30&$expand=Orders&$filter=Status eq 'Active'",
    ]
    
    for query in valid_queries:
        result = validator.validate(query)
        print_result(query, result)


def example_2_limit_violations():
    """Example 2: Queries that violate retrieval limits."""
    print("\n" + "🔴 "*40)
    print("EXAMPLE 2: Limit Violations")
    print("🔴 "*40)
    
    # Use strict production config (max 500 rows)
    validator = ODataQueryValidator(config=ValidatorConfig.for_production())
    
    limit_violation_queries = [
        "Users?$top=1000",  # Exceeds production limit of 500
        "Orders?$top=600",  # Also exceeds limit
        "Products?$filter=Price gt 0",  # Missing required $top
        "Customers?$top=-10",  # Negative value
        "Items?$top=abc",  # Non-numeric value
    ]
    
    for query in limit_violation_queries:
        result = validator.validate(query)
        print_result(query, result)


def example_3_security_violations():
    """Example 3: Queries with security issues."""
    print("\n" + "🚨 "*40)
    print("EXAMPLE 3: Security Violations")
    print("🚨 "*40)
    
    validator = ODataQueryValidator()
    
    unsafe_queries = [
        ("Mutation attempt", "Users?$filter=Name eq 'test' DELETE"),
        ("SQL injection", "Users?$filter=Name eq 'admin' OR '1'='1'"),
        ("Script injection", "Users?$filter=Name eq '<script>alert(1)</script>'"),
        ("Union injection", "Users?$filter=Name eq 'x' UNION SELECT * FROM Passwords"),
        ("Command injection", "Users?$filter=Name eq 'test'; exec('rm -rf /')"),
    ]
    
    for name, query in unsafe_queries:
        print(f"\n{'='*80}")
        print(f"Attack Type: {name}")
        result = validator.validate(query)
        print_result(query, result)
    
    # Also test HTTP method violations
    print(f"\n{'='*80}")
    print("HTTP Method Violations:")
    print(f"{'='*80}")
    
    for method in ["POST", "PUT", "PATCH", "DELETE"]:
        result = validator.validate("Users", http_method=method)
        print(f"\nMethod: {method}")
        print(f"Status: {'✓ VALID' if result.is_valid else '✗ INVALID'}")
        if not result.is_valid:
            print(f"Error: {result.critical_issues[0].message}")


def example_4_complexity_warnings():
    """Example 4: Queries with complexity warnings."""
    print("\n" + "🟡 "*40)
    print("EXAMPLE 4: Complexity Warnings")
    print("🟡 "*40)
    
    validator = ODataQueryValidator()
    
    complex_queries = [
        # High retrieval count (but within limits)
        "Users?$top=900&$select=Name",
        
        # Deep expand nesting
        "Orders?$top=10&$expand=Items($expand=Product($expand=Category($expand=Supplier)))",
        
        # Complex filter
        "Products?$top=50&$filter=(Price gt 100 and Price lt 500) or (Category eq 'A' and InStock eq true) or (Brand eq 'X' and Rating gt 4)",
        
        # No limit + complex operations
        "Orders?$expand=Items($expand=Product)&$filter=TotalAmount gt 1000&$orderby=OrderDate desc",
        
        # High cost combination
        "Customers?$top=500&$expand=Orders($expand=Items($expand=Product))&$filter=contains(Name, 'Corp') and Status eq 'Active'",
    ]
    
    for query in complex_queries:
        result = validator.validate(query)
        print_result(query, result)


def example_5_configuration_comparison():
    """Example 5: Same query with different configurations."""
    print("\n" + "⚙️  "*40)
    print("EXAMPLE 5: Configuration Comparison")
    print("⚙️  "*40)
    
    query = "Users?$top=600&$expand=Orders($expand=Items)"
    
    configs = {
        "Production (Strict)": ValidatorConfig.for_production(),
        "Development (Permissive)": ValidatorConfig.for_development(),
        "Custom (Medium)": ValidatorConfig(
            max_retrieval_limit=750,
            max_expand_depth=2,
            require_top_limit=True
        ),
    }
    
    print(f"\nTesting query: {query}")
    print(f"\n{'='*80}")
    
    for config_name, config in configs.items():
        validator = ODataQueryValidator(config=config)
        result = validator.validate(query)
        
        print(f"\nConfiguration: {config_name}")
        print(f"  Max Retrieval Limit: {config.max_retrieval_limit}")
        print(f"  Max Expand Depth: {config.max_expand_depth}")
        print(f"  Status: {'✓ VALID' if result.is_valid else '✗ INVALID'}")
        
        if result.issues:
            print(f"  Issues: {len(result.issues)}")
            for issue in result.issues:
                print(f"    • [{issue.severity.upper()}] {issue.message}")


def example_6_realistic_scenarios():
    """Example 6: Realistic business scenarios."""
    print("\n" + "📊 "*40)
    print("EXAMPLE 6: Realistic Business Scenarios")
    print("📊 "*40)
    
    validator = ODataQueryValidator()
    
    scenarios = [
        (
            "Dashboard - Active Users",
            "Users?$top=100&$filter=Status eq 'Active' and LastLogin gt 2024-01-01&$select=Name,Email,LastLogin&$orderby=LastLogin desc"
        ),
        (
            "Report - Monthly Sales",
            "Orders?$top=500&$filter=OrderDate ge 2024-05-01 and OrderDate lt 2024-06-01&$select=OrderId,CustomerName,TotalAmount&$orderby=OrderDate"
        ),
        (
            "Search - Product Lookup",
            "Products?$top=20&$filter=contains(Name, 'laptop') and Price lt 2000&$select=Name,Price,InStock&$orderby=Price"
        ),
        (
            "Detail View - Order with Items",
            "Orders?$top=1&$filter=OrderId eq 12345&$expand=OrderItems($select=ProductName,Quantity,Price)"
        ),
        (
            "Analytics - Customer Orders (Too Complex)",
            "Customers?$top=1000&$expand=Orders($expand=OrderItems($expand=Product($expand=Category)))&$filter=TotalSpent gt 10000"
        ),
    ]
    
    for scenario_name, query in scenarios:
        print(f"\n{'='*80}")
        print(f"Scenario: {scenario_name}")
        result = validator.validate(query)
        print_result(query, result)


def example_7_validation_result_api():
    """Example 7: Using the ValidationResult API."""
    print("\n" + "🔧 "*40)
    print("EXAMPLE 7: ValidationResult API Usage")
    print("🔧 "*40)
    
    validator = ODataQueryValidator()
    query = "Users?$top=10000&$expand=Orders($expand=Items($expand=Product($expand=Category)))"
    
    result = validator.validate(query)
    
    print(f"\nQuery: {query}\n")
    
    # Basic properties
    print("Basic Properties:")
    print(f"  is_valid: {result.is_valid}")
    print(f"  query: {result.query}")
    print(f"  estimated_cost_score: {result.estimated_cost_score}")
    print(f"  estimated_row_count: {result.estimated_row_count}")
    
    # Issue categorization
    print(f"\nIssue Categorization:")
    print(f"  Total issues: {len(result.issues)}")
    print(f"  Critical issues: {len(result.critical_issues)}")
    print(f"  Warnings: {len(result.warnings)}")
    print(f"  has_critical_issues: {result.has_critical_issues}")
    print(f"  has_warnings: {result.has_warnings}")
    
    # Error message
    print(f"\nFormatted Error Message:")
    error_msg = result.get_error_message()
    if error_msg:
        print(error_msg)
    else:
        print("  (None - query is valid)")
    
    # Dictionary serialization
    print(f"\nSerialized to Dict:")
    result_dict = result.to_dict()
    import json
    print(json.dumps(result_dict, indent=2))


def main():
    """Run all examples."""
    print("\n" + "🚀 "*40)
    print("OData Query Validator - Standalone Examples")
    print("🚀 "*40)
    
    examples = [
        ("Valid Queries", example_1_valid_queries),
        ("Limit Violations", example_2_limit_violations),
        ("Security Violations", example_3_security_violations),
        ("Complexity Warnings", example_4_complexity_warnings),
        ("Configuration Comparison", example_5_configuration_comparison),
        ("Realistic Scenarios", example_6_realistic_scenarios),
        ("ValidationResult API", example_7_validation_result_api),
    ]
    
    for i, (name, example_func) in enumerate(examples, 1):
        print(f"\n\n{'#'*80}")
        print(f"# Example {i}: {name}")
        print(f"{'#'*80}")
        
        try:
            example_func()
        except Exception as e:
            print(f"\n❌ Example failed with error: {e}")
            import traceback
            traceback.print_exc()
    
    print("\n\n" + "✅ "*40)
    print("All examples completed!")
    print("✅ "*40 + "\n")


if __name__ == "__main__":
    main()
